# Data Preparation

The notebook will show all the options available for preparing the data before sending it to the model.

In [2]:
import numpy as np

from sckitflow.data import DataManager
from sckitflow.data.sim import get_dummy_adata
from sckitflow.dataset.toy_data import get_toy_dataset
from sklearn.decomposition import PCA

## Preliminaries

### Dummy Datasets

`sckitflow` allows you to create different types of dummy datasets using `sklearn.datasets` and wraps it in `Anndata` format using the function `get_toy_dataset(name, **kwargs)` where `name` refers to the type of dataset to be created and `**kwargs` are the additional parameters specific to dataset type. 

The following code block shows all the possible dataset types for creating dummy data

In [5]:
blobs_data = get_toy_dataset("blobs").adata
checkerboard_data = get_toy_dataset("checkerboard").adata
circles_data = get_toy_dataset("circles").adata
moons_data = get_toy_dataset("moons").adata
s_curve_data = get_toy_dataset("s_curve").adata
swiss_roll_data = get_toy_dataset("swiss_roll").adata

Investigating the data, we can see that the anndata structure varies in terms of fields with each data type. However, all of them have `adata.uns['dataset_info]` which contains all the parameters used in dataset creation. The default parameters remain the same from `sklearn.datasets` default values.

In [6]:
print(blobs_data)
print(checkerboard_data)

AnnData object with n_obs × n_vars = 1000 × 2
    uns: 'dataset_info'
    obsm: 'Y'
AnnData object with n_obs × n_vars = 1000 × 2
    uns: 'dataset_info'
    obsm: 'row_cluster'
    varm: 'col_cluster'


In [7]:
blobs_data.uns["dataset_info"]

{'name': 'blobs',
 'random_state': 42,
 'No of centers': 3,
 'cluster_std': 1.0,
 'center_box': (-10.0, 10.0),
 'shuffle': True,
 'centers': {'0': array([-2.50919762,  9.01428613]),
  '1': array([4.63987884, 1.97316968]),
  '2': array([-6.87962719, -6.88010959])}}

### PCA Representation

Preprocessing such as PCA is **not** performed by `sckitflow` — the caller computes the representation with their tool of choice and stores it in `.obsm`. Here we use scikit-learn's `PCA` and later point `DataManager` at it via `sample_rep`. Fitted attributes such as `pca.components_`, `pca.explained_variance_`, `pca.mean_`, and `pca.n_components_` live on the estimator.

In [8]:
blobs_data = get_toy_dataset("blobs", n_features=200, centers=5).adata

pca = PCA(n_components=50)
blobs_data.obsm["X_pca"] = pca.fit_transform(blobs_data.X)

print(f"Original Shape: {blobs_data.X.shape}")
print(f"New Shape: {blobs_data.obsm['X_pca'].shape}")

Original Shape: (1000, 200)
New Shape: (1000, 50)


## DataManager

### Basic Initialization

Creating Dummy data with 5000 samples and 200 features

In [ ]:
dummy_data = get_toy_dataset("blobs", n_samples=5000, n_features=200, centers=5).adata
pca = PCA(n_components=50)
dummy_data.obsm["X_pca"] = pca.fit_transform(dummy_data.X)
print(dummy_data)
print(dummy_data.obsm["X_pca"].shape)

AnnData object with n_obs × n_vars = 5000 × 200
    uns: 'dataset_info'
    obsm: 'Y', 'X_pca'
(5000, 50)


Initializing DataManager and compiling the data

In [18]:
dm = DataManager()
train_collection = dm.compile_adata(dummy_data)
print(train_collection)

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 35.46it/s]

NestedData(mapping={(): NestedData(mapping={(): MatchedData:
 * (target) -> 	DistributionData:
	 * n_obs=5000
	 state=StateData(n_obs=5000, spatial_dims=(200,))
	 target=None
	 condition=None
	 groups=CategoricalData(n_obs=5000, n_vars=0, columns=[], repr_dict_keys=[], categorical_encoders_keys=[])
	 source_coupling=CouplingData(n_obs=5000, linear(spatial_dims=(200,)), quadratic=None)
	 target_coupling=CouplingData(n_obs=5000, linear(spatial_dims=(200,)), quadratic=None)})})


Initializing DataManager with the PCA representation and compiling the data. `DataManager` checks for the key in `.obsm`. It defaults to `.X` if key not found.

In [17]:
dm1 = DataManager(sample_rep="X_pca")
train_collection1 = dm1.compile_adata(dummy_data)
print(train_collection1)

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 86.12it/s]

NestedData(mapping={(): NestedData(mapping={(): MatchedData:
 * (target) -> 	DistributionData:
	 * n_obs=5000
	 state=StateData(n_obs=5000, spatial_dims=(50,))
	 target=None
	 condition=None
	 groups=CategoricalData(n_obs=5000, n_vars=0, columns=[], repr_dict_keys=[], categorical_encoders_keys=[])
	 source_coupling=CouplingData(n_obs=5000, linear(spatial_dims=(50,)), quadratic=None)
	 target_coupling=CouplingData(n_obs=5000, linear(spatial_dims=(50,)), quadratic=None)})})


### Advanced Initialization

The initialization of `Datamanager` can be classified into 5 groups -

* Group Data
* State Data
* Coupling Data
* Condition Data
* Response Data

#### Group Data

This group of parameters control how the oberservation metadata is turned into numerical encoding. The parameters are -

* `groups`: Collection of `.obs` identifiers used to define grouping.


* `groups_reps`: Dictionary mapping `.obs` identifiers to pre defined numerical encoding which are stored in `.uns`


* `groups_encoding`: This parameter is used if the encoding have not been created and stored in `.uns` and we want to create the encoding now. The input is a dictionary mapping the `groups` identifiers to an encoder. Each group column takes either `groups_reps` or `groups_encoding` — never both.


* __NOTE__ The strings `"label"` and `"one-hot"` are shorthand for the two parameter-free encoders. For anything parameterized, pass an encoder instance from `sckitflow.data.group_encoders` (`from sckitflow.data import group_encoders`) — `Label`, `OneHot`, `Identity`, `Log1p`, `Affine`. These are frozen dataclasses of plain scalars rather than callables, which is what lets a whole `DataManager` be serialized. Use them to pin a vocabulary, as in `OneHot(categories=("control", "drugA"))`, so a round-tripped config reproduces identical columns; unknown categories then raise instead of being silently dropped.

In [22]:
# Creating Dummy data
dummy_data = get_toy_dataset("blobs", n_samples=5000, n_features=200, centers=5).adata

# Creating dummy .obs columns
drug_labels = ["drugA", "control"]
ko_labels = ["koA", "control"]
dummy_data.obs["drug"] = np.random.choice(drug_labels, size=dummy_data.n_obs)
dummy_data.obs["ko"] = np.random.choice(ko_labels, size=dummy_data.n_obs)

In [23]:
dm = DataManager(
    groups=["drug", "ko"],
    groups_encoding={"drug": "one-hot", "ko": "label"},
)
train_collection = dm.compile_adata(dummy_data, sort=True)
print(train_collection)

100%|██████████| 1/1 [00:00<00:00, 208.83it/s]

NestedData(mapping={('control', 'control'): NestedData(mapping={(): MatchedData:
 * (target) -> 	DistributionData:
	 * n_obs=1260
	 state=StateData(n_obs=1260, spatial_dims=(200,))
	 target=None
	 condition=None
	 groups=CategoricalData(n_obs=1260, n_vars=2, columns=['drug', 'ko'], repr_dict_keys=[], categorical_encoders_keys=['drug', 'ko'])
	 source_coupling=CouplingData(n_obs=1260, linear(spatial_dims=(200,)), quadratic=None)
	 target_coupling=CouplingData(n_obs=1260, linear(spatial_dims=(200,)), quadratic=None)}), ('drugA', 'control'): NestedData(mapping={(): MatchedData:
 * (target) -> 	DistributionData:
	 * n_obs=1266
	 state=StateData(n_obs=1266, spatial_dims=(200,))
	 target=None
	 condition=None
	 groups=CategoricalData(n_obs=1266, n_vars=2, columns=['drug', 'ko'], repr_dict_keys=[], categorical_encoders_keys=['drug', 'ko'])
	 source_coupling=CouplingData(n_obs=1266, linear(spatial_dims=(200,)), quadratic=None)
	 target_coupling=CouplingData(n_obs=1266, linear(spatial_dims=(200

We can see that the data is split based on the defined groups. Here, the data is split into 4 groups `(drugA, koA), (control, koA), (drugA, control), (control, control)`.

If for any reason you want to dig deeper into the data structure, you can use `.mapping`

In [24]:
train_collection.mapping[("control", "koA")]

NestedData(mapping={(): MatchedData:
 * (target) -> 	DistributionData:
	 * n_obs=1211
	 state=StateData(n_obs=1211, spatial_dims=(200,))
	 target=None
	 condition=None
	 groups=CategoricalData(n_obs=1211, n_vars=2, columns=['drug', 'ko'], repr_dict_keys=[], categorical_encoders_keys=['drug', 'ko'])
	 source_coupling=CouplingData(n_obs=1211, linear(spatial_dims=(200,)), quadratic=None)
	 target_coupling=CouplingData(n_obs=1211, linear(spatial_dims=(200,)), quadratic=None)})

In [25]:
# If for some reason you want to access the data of a specific group combination, you can do so like this:
train_collection.mapping[("control", "koA")].mapping[()]["target"].state_data.X.shape

(1211, 200)

We can use `DataManager.groups_data_schema` to look at our variables

In [26]:
print(dm.groups_data_schema.groups)
print(dm.groups_data_schema.groups_encoders)
print(dm.groups_data_schema.groups_reps)

['drug', 'ko']
{'drug': OneHot(categories=None), 'ko': Label(classes=None)}
{}


#### State Data

This is just the basic initialization. The only parameter here is `sample_rep` by which you can choose which representation you want to use as input. The str value will be searched in `.obs` and if not found, it reverts to `.X`

In [18]:
dummy_data = get_toy_dataset("blobs", n_samples=5000, n_features=200, centers=5).adata
pca = PCA(n_components=50)
dummy_data.obsm["X_pca"] = pca.fit_transform(dummy_data.X)

dm = DataManager(sample_rep="X_pca")
train_collection = dm.compile_adata(dummy_data)
print(train_collection)

100%|██████████| 1/1 [00:00<00:00, 107.60it/s]

NestedData(mapping={(): NestedData(mapping={(): MatchedData:
 * (target) -> 	DistributionData:
	 * n_obs=5000
	 state=StateData(n_obs=5000, spatial_dims=(50,))
	 target=None
	 condition=None
	 groups=CategoricalData(n_obs=5000, n_vars=0, columns=[], repr_dict_keys=[], categorical_encoders_keys=[])
	 source_coupling=CouplingData(n_obs=5000, linear(spatial_dims=(50,)), quadratic=None)
	 target_coupling=CouplingData(n_obs=5000, linear(spatial_dims=(50,)), quadratic=None)})})


In [19]:
dm.state_data_schema.sample_rep

'X_pca'

#### Coupling Data

This group of parameters control how the source and the target data distribution is coupled. The parameters are-

1. `sample_rep`: To choose the sourse representation

2. `target_rep`: To choose the target representation

3. `n_shared_dims`: If for some reason, the source and target embedding is not in the same latent space, then we have the option to choose the first n dimesions from both embeddings to sort of create a latent space.

In [3]:
n_obs_pert = 5000
n_obs_ctrl = 200
adata = get_dummy_adata(n_obs_pert=n_obs_pert, n_obs_ctrl=n_obs_ctrl)

C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3312.0_x64__qbz5n2kfra8p0\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3312.0_x64__qbz5n2kfra8p0\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [9]:
dm = DataManager(
    sample_rep="X_tgt",
    source_rep="X_src",
    n_shared_dims=10,
)
train_collection = dm.compile_adata(adata)
train_collection

100%|██████████| 1/1 [00:00<00:00, 95.89it/s]


NestedData(mapping={(): NestedData(mapping={(): MatchedData:
 * (target) -> 	DistributionData:
	 * n_obs=5200
	 state=StateData(n_obs=5200, spatial_dims=(22,))
	 target=None
	 condition=None
	 groups=CategoricalData(n_obs=5200, n_vars=0, columns=[], repr_dict_keys=[], categorical_encoders_keys=[])
	 source_coupling=CouplingData(n_obs=5200, linear(spatial_dims=(10,)), quadratic(spatial_dims=(6,)))
	 target_coupling=CouplingData(n_obs=5200, linear(spatial_dims=(10,)), quadratic(spatial_dims=(12,)))})})

In [7]:
print(dm.coupling_data_schema.source_rep)
print(dm.coupling_data_schema.target_rep)
print(dm.coupling_data_schema.n_shared_dims)
print(dm.coupling_data_schema.has_incomparable_spaces)

X_src
X_tgt
10
True


#### Condition Data

In [27]:
# regenerate data with a categorical condition column for this section
dummy_data = get_toy_dataset("blobs", n_samples=5000, n_features=200, centers=5).adata
dummy_data.obs["drug"] = np.random.choice(["drugA", "control"], size=dummy_data.n_obs)
dummy_data

AnnData object with n_obs × n_vars = 5000 × 200
    obs: 'drug', 'ko'
    uns: 'dataset_info'
    obsm: 'Y'

In [28]:
dm = DataManager(target_categorical_covs_dict={"drug": "one-hot"})
train_collection = dm.compile_adata(dummy_data)

100%|██████████| 1/1 [00:00<00:00, 42.28it/s]


In [30]:
dm1 = DataManager()
train_collection1 = dm1.compile_adata(dummy_data)

100%|██████████| 1/1 [00:00<00:00, 38.93it/s]


In [29]:
train_collection

NestedData(mapping={(): NestedData(mapping={(): MatchedData:
 * (target) -> 	DistributionData:
	 * n_obs=5000
	 state=StateData(n_obs=5000, spatial_dims=(200,))
	 target=MixedTypeData(n_obs=5000, categorical(n_vars=1, columns=['drug']), continuous=None
	 condition=None
	 groups=CategoricalData(n_obs=5000, n_vars=0, columns=[], repr_dict_keys=[], categorical_encoders_keys=[])
	 source_coupling=CouplingData(n_obs=5000, linear(spatial_dims=(200,)), quadratic=None)
	 target_coupling=CouplingData(n_obs=5000, linear(spatial_dims=(200,)), quadratic=None)})})

In [31]:
train_collection1

NestedData(mapping={(): NestedData(mapping={(): MatchedData:
 * (target) -> 	DistributionData:
	 * n_obs=5000
	 state=StateData(n_obs=5000, spatial_dims=(200,))
	 target=None
	 condition=None
	 groups=CategoricalData(n_obs=5000, n_vars=0, columns=[], repr_dict_keys=[], categorical_encoders_keys=[])
	 source_coupling=CouplingData(n_obs=5000, linear(spatial_dims=(200,)), quadratic=None)
	 target_coupling=CouplingData(n_obs=5000, linear(spatial_dims=(200,)), quadratic=None)})})

#### Response Data